# VMIM path — CNN compressor on raw cubes, VMIM objective
The 3D CNN compressor is trained with the VMIM objective (variational MI maximization), then the shared NLE + MCMC machinery is applied.

## Setup

In [ ]:
import os, glob, subprocess
from pathlib import Path
from IPython.display import Image, display
import pandas as pd, yaml

os.environ["MPLBACKEND"] = "Agg"  # headless plotting for the CLI tools
REPO = subprocess.check_output(["git", "rev-parse", "--show-toplevel"]).decode().strip()
os.chdir(REPO)
print("repo:", REPO)


def sh(cmd):
    """Run a shell command, echoing it first; its output streams into the cell."""
    print("$", cmd, flush=True)
    return subprocess.run(cmd, shell=True)


def show(path, width=1000):
    """Display a PNG, or rasterize + display page 1 of a PDF (needs pdftoppm)."""
    p = str(path)
    if p.endswith(".png") and os.path.exists(p):
        display(Image(p, width=width))
        return
    if p.endswith(".pdf") and os.path.exists(p):
        prev = p[:-4] + "_preview"
        if os.system(
            f'pdftoppm -png -r 150 -singlefile "{p}" "{prev}" 2>/dev/null'
        ) == 0 and os.path.exists(prev + ".png"):
            display(Image(prev + ".png", width=width))
            return
    print("figure not found:", p)

## Config

In [ ]:
# ---- path identity -------------------------------------------------------
PATHDIR = "vmim"  # figures -> notebooks/figs_<PATHDIR>/
CONFIG = [
    "configs_seeds/vmim_/arm_cnn_vmim_jitter_n1.yaml",
]  # arm config(s) to run
# other vmim arms: configs_seeds/vmim_/arm_cnn_vmim_jitter_n2.yaml

# ---- sweep knobs ---------------------------------------------------------
SEEDS = [0]  # compressor.init_seed values, one run each
FAMILY = "nsf"  # NLE density family/families: gmm | maf | nsf
SCOPE = "std"  # t-scaling(s): std (standardized) | raw
STAGES = "123"  # 1=compressor, 2=NLE, 3=MCMC
NAMES = "Fx tau rHS Mmin"  # physical parameter names (plot legends)
NOTE = "vmim path — CNN compressor, VMIM objective"
PUSH = False  # git-commit + push experiments/manifest.csv after launch

# ---- figure sub-dirs for THIS path --------------------------------------
FIGS = Path("notebooks") / f"figs_{PATHDIR}"
DIR_COMP, DIR_NLE, DIR_SBC = FIGS / "compressor", FIGS / "nle", FIGS / "sbc"
for d in (DIR_COMP, DIR_NLE, DIR_SBC):
    d.mkdir(parents=True, exist_ok=True)


# ---- resolve each run's directory from its own YAML ---------------------
def nle_dir(config, s):
    c = yaml.safe_load(open(config))
    return f"{c['scratch_root']}/{c['arm_name']}_s{s}/nle"


def cfg_tag(config):
    return Path(config).stem.replace("arm_", "")


print("path    :", PATHDIR, "| figures ->", FIGS)
print("family  :", FAMILY, "| scope:", SCOPE, "| stages:", STAGES, "| seeds:", SEEDS)
for config in CONFIG:
    for s in SEEDS:
        d = nle_dir(config, s)
        print(f"  {cfg_tag(config)} s{s}: {d}  {'OK' if os.path.isdir(d) else 'missing'}")

## 1 · Run experiment  (stages 1→2→3, SLURM)

In [ ]:
# Launch stages 1->2->3 for each config on SLURM. This SUBMITS and returns at once;
# wait for the jobs to finish (squeue below) before running the plot cells.
seeds = " ".join(map(str, SEEDS))
push = "--push" if PUSH else ""
for config in CONFIG:
    sh(
        f'python experiments/run_sweep.py "{config}" --seeds {seeds} '
        f'--family "{FAMILY}" --scope "{SCOPE}" --stages {STAGES} '
        f'--note "{NOTE}" {push}'
    )

In [ ]:
sh("squeue --me")

## 2 · Compressor training

In [ ]:
# Compressor training: RF-R^2 / conditional-sigma / loss, every (config, seed) overlaid.
arms = " ".join(f"{cfg_tag(c)}_s{s}={nle_dir(c, s)}" for c in CONFIG for s in SEEDS)
fig = DIR_COMP / f"{PATHDIR}_compressor.pdf"
sh(f'python tools/plot_training_compressor.py {arms} --names {NAMES} --out "{fig}" --png')
show(fig.with_suffix(".png"), width=1100)

## 3 · NLE training

In [ ]:
# NLE training: train (solid) / val (dashed) loss per family, all seeds overlaid.
seeds = " ".join(map(str, SEEDS))
cfgs = " ".join(f'"{c}"' for c in CONFIG)
fig = DIR_NLE / f"{PATHDIR}_nle.png"
sh(f'python tools/plot_training_nle.py {cfgs} --seeds {seeds} --models {FAMILY} --out "{fig}"')
show(fig, width=1000)

## 4 · SBC campaign

In [ ]:
# SBC campaign (stage 4 / eval.py): rank histograms, coverage metrics, overlay corner.
# One --item per (config, family, scope); seeds averaged with --all-seeds.
items = [
    f"{c}|{fam}|{sc}|{cfg_tag(c)} {fam}/{sc}"
    for c in CONFIG
    for fam in FAMILY.split()
    for sc in SCOPE.split()
]
item_args = " ".join(f'--item "{it}"' for it in items)
seeds = " ".join(map(str, SEEDS))
sh(
    f'CUDA_VISIBLE_DEVICES="" python eval.py {item_args} --all-seeds --seeds {seeds} '
    f'--dlogp 10 --corner-sim median --out "{DIR_SBC}"'
)
# Heavy campaign? submit instead of running inline:
#   sbatch slurm/stage4_nb.sbatch raw {item_args} --all-seeds --seeds {seeds} --out {DIR_SBC}

In [ ]:
m = DIR_SBC / "metrics.csv"
if m.exists():
    display(pd.read_csv(m))
for pdf in sorted(glob.glob(str(DIR_SBC / "*.pdf"))):
    print(os.path.basename(pdf))
    show(pdf, width=850)